# 并发编排的旅行推荐

本笔记本演示了使用 Microsoft Agent Framework 的**并发编排**。我们将构建一个旅行推荐系统，利用三个专业代理并行工作，提供全面的旅行洞察。

## 您将学习：
1. **并发编排**：以扇出/扇入模式并行运行多个代理
2. **ConcurrentBuilder**：构建并发工作流的高级 API
3. **旅行推荐**：三个专业代理协同工作
4. **默认聚合**：合并多个代理的响应
5. **性能优势**：并行执行与顺序处理的对比

## 三个专业代理：

1. **景点代理**：旅游景点、活动、地标
2. **美食代理**：当地美食、餐厅、饮食体验
3. **历史代理**：历史事实、文化意义、背景


In [1]:
# 导入异步编程支持库，后面会用 `await` 运行工作流。
import asyncio
# 导入 JSON 库，用于处理模型返回的 JSON 文本。
import json
# 导入环境变量库，用来读取 API Key。
import os
# 导入时间库，用来做并发和串行的耗时对比。
import time
# 导入类型标注工具，让代码更容易理解。
from typing import Any

# 从 agent framework 中导入 Agent 和 Message 基础类型。
from agent_framework import Agent, Message
# 导入并发和顺序工作流构建器。
from agent_framework.orchestrations import ConcurrentBuilder, SequentialBuilder
# 导入兼容 OpenAI Chat Completions 的客户端。
from agent_framework.openai import OpenAIChatCompletionClient
# 导入 dotenv，用于加载 .env 文件。
from dotenv import load_dotenv
# 导入 notebook 的 HTML 展示工具。
from IPython.display import HTML, display
# 导入 BaseModel 和 Field，用于定义结构化输出模型。
from pydantic import BaseModel, Field

# 打印提示信息，确认导入成功。
print("All imports successful!")


All imports successful!


## 第一步：定义用于结构化输出的 Pydantic 模型

这些模型定义了每个专用代理将返回的模式。这确保了所有代理的响应一致且可解析。


## 第一步：定义用于结构化输出的 Pydantic 模型

这些模型定义了每个专用代理将返回的模式。这确保了所有代理的响应一致且可解析。


In [2]:
# 定义景点推荐的数据结构，约束景点代理的输出格式。
class AttractionsRecommendation(BaseModel):
    """用于承载景点和活动推荐结果。"""

    # 旅行目的地名称。
    destination: str = "目的主机不可知"
    # 重点景点列表；如果模型漏掉该字段，就使用空列表兜底。
    top_attractions: list[str] = Field(default_factory=list)
    # 推荐活动列表；如果模型漏掉该字段，就使用空列表兜底。
    activities: list[str] = Field(default_factory=list)
    # 最佳游玩时间；给一个默认说明，避免校验失败。
    best_time_to_visit: str = "最佳时间未指定。"
    # 交通建议；给一个默认说明，避免校验失败。
    transportation_tips: str = "交通建议未指定。"


# 定义美食推荐的数据结构，约束餐饮代理的输出格式。
class DiningRecommendation(BaseModel):
    """用于承载饮食和餐厅推荐结果。"""

    # 旅行目的地名称。
    destination: str = "目的主机不可知"
    # 当地菜系简介。
    local_cuisine: str = "没有指定当地美食。"
    # 必吃菜品列表；如果模型漏掉该字段，就使用空列表兜底。
    must_try_dishes: list[str] = Field(default_factory=list)
    # 推荐餐厅列表；如果模型漏掉该字段，就使用空列表兜底。
    recommended_restaurants: list[str] = Field(default_factory=list)
    # 饮食体验列表；如果模型漏掉该字段，就使用空列表兜底。
    food_experiences: list[str] = Field(default_factory=list)
    # 用餐礼仪说明。
    dining_etiquette: str = "用餐礼仪未指定。"


# 定义历史文化推荐的数据结构，约束历史代理的输出格式。
class HistoryRecommendation(BaseModel):
    """用于承载历史和文化背景结果。"""

    # 旅行目的地名称。
    destination: str = "未知目的地"
    # 历史意义概述。
    historical_significance: str = "历史意义未指定。"
    # 文化亮点列表；如果模型漏掉该字段，就使用空列表兜底。
    cultural_highlights: list[str] = Field(default_factory=list)
    # 重要历史时期列表；如果模型漏掉该字段，就使用空列表兜底。
    important_periods: list[str] = Field(default_factory=list)
    # 文化体验列表；如果模型漏掉该字段，就使用空列表兜底。
    cultural_experiences: list[str] = Field(default_factory=list)
    # 有趣事实列表；如果模型漏掉该字段，就使用空列表兜底。
    interesting_facts: list[str] = Field(default_factory=list)


## 第2步：加载环境变量

按照中间件笔记本的相同模式配置 LLM 客户端（GitHub Models 或 OpenAI）。


In [3]:
# Load environment variables
# 从 `.env` 文件加载环境变量配置。
load_dotenv()

# Configure DashScope Qwen via the OpenAI-compatible Chat Completions client
# 创建兼容 OpenAI 接口的聊天客户端，用来连接模型服务。
chat_client = OpenAIChatCompletionClient(
    base_url="https://models.inference.ai.azure.com/",  # DashScope OpenAI兼容接口
    api_key=os.environ.get("GITHUB_TOKEN"),                  # DashScope API Key
    model="gpt-4.1-mini"                                              # 使用的模型名称
)

print("Chat client configured with DashScope gpt-4.1-mini")


Chat client configured with DashScope gpt-4.1-mini


## 第三步：创建三个专业旅行代理人


In [4]:
# 创建景点专家代理，负责输出景点和活动建议。
attractions_agent = Agent(
    # 把模型客户端注入给当前代理。
    client=chat_client,
    # 编写给模型的系统提示词，明确输出任务和字段要求。
    instructions=(
        """
        你是一名旅游专家，专门为用户推荐旅游景点和旅行活动。

        当用户提供一个旅行目的地时，请为其提供全面的旅行建议，包括：
        - 热门旅游景点；
        - 推荐旅行活动；
        - 最佳旅游时间；
        - 当地交通出行建议。

        请重点介绍当地最受欢迎的地标景点、独特的旅行体验以及实用的旅行建议。

        **你必须仅返回合法的 JSON 格式，不要输出任何其他内容。**

        返回的 JSON 必须始终包含以下字段，并且字段名称必须完全一致：

        - destination
        - top_attractions
        - activities
        - best_time_to_visit
        - transportation_tips
        """
    ),
    # 设置代理名称，便于后面区分是谁返回的消息。
    name="attractions_agent",
    # 指定结构化输出模型。
    default_options={"response_format": AttractionsRecommendation},
)

# 创建美食专家代理，负责输出餐饮建议。
dining_agent = Agent(
    # 把模型客户端注入给当前代理。
    client=chat_client,
    # 编写给模型的系统提示词，明确输出任务和字段要求。
    instructions=(
        """
        你是一名美食专家，专门推荐当地美食和用餐体验。

        当用户提供一个旅行目的地时，请为其提供当地的美食推荐，包括：
        - 当地菜系简介；
        - 必吃菜品列表；
        - 推荐餐厅列表；
        - 独特的饮食体验。

        请包含用餐礼仪和当地饮食文化习俗。

        **你必须仅返回合法的 JSON 格式，不要输出任何其他内容。**

        返回的 JSON 必须始终包含以下字段，并且字段名称必须完全一致：

        - destination
        - local_cuisine
        - must_try_dishes
        - recommended_restaurants
        - food_experiences
        - dining_etiquette
        """
    ),
    # 设置代理名称，便于后面区分是谁返回的消息。
    name="dining_agent",
    # 指定结构化输出模型。
    default_options={"response_format": DiningRecommendation},
)

# 创建历史文化专家代理，负责输出历史和文化背景。
history_agent = Agent(
    # 把模型客户端注入给当前代理。
    client=chat_client,
    # 编写给模型的系统提示词，明确输出任务和字段要求。
    instructions=(
        """
        你是一名历史和文化专家。
        当用户提供一个旅行目的地时，请提供历史背景、文化意义、重要地标、当地习俗和文化洞察。
        重点关注引人入胜的历史事实和文化理解。
        **你必须仅返回合法的 JSON 格式，不要输出任何其他内容。**
        返回的 JSON 必须始终包含以下字段，并且字段名称必须完全一致：
        - destination
        - historical_significance
        - cultural_highlights
        - important_periods
        - cultural_experiences
        - interesting_facts
        """
    ),
    # 设置代理名称，便于后面区分是谁返回的消息。
    name="history_agent",
    # 指定结构化输出模型。
    default_options={"response_format": HistoryRecommendation},
)

# 在 notebook 中展示代理创建成功的说明块。
display(HTML("""
<div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
    <strong>✅ 已创建 3 个专业智能体：</strong>
    <ul style='margin: 10px 0 0 0;'>
        <li><strong>attractions_agent</strong> - 负责旅游景点与旅行活动推荐</li>
        <li><strong>dining_agent</strong> - 负责美食与餐饮推荐</li>
        <li><strong>history_agent</strong> - 负责历史文化与背景介绍</li>
    </ul>
</div>
"""))


# 第四步：构建并发工作流

ConcurrentBuilder 创建的工作流可以：
1. **分发**相同的输入到所有三个代理同时运行（扇出）
2. **并行运行代理**以提高性能
3. **汇总**所有响应为一个单一输出（扇入）
4. **返回**所有代理的合并 ChatMessage 列表


In [5]:
# 创建并发工作流构建器，让多个 Agent 并行执行。
workflow = ConcurrentBuilder(
    participants=[attractions_agent, dining_agent, history_agent],
# 根据前面配置生成最终可运行的工作流对象。
).build()

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>🎉 并发工作流构建成功！</h3>

    <p style='margin: 0; line-height: 1.6;'>
        <strong>系统架构：</strong><br>
        • 用户输入 → <strong>调度器（Dispatcher）</strong>（任务分发 / Fan-out）<br>
        • <strong>3 个智能体</strong>并行执行（景点推荐、美食推荐、历史文化）<br>
        • <strong>聚合器（Aggregator）</strong>汇总各智能体的结果（结果汇聚 / Fan-in）<br>
        • 输出 → 综合旅行推荐结果
    </p>
</div>
"""))


## 第五步：测试案例 1 - 东京旅行推荐

让我们以东京为目的地测试我们的并发工作流程。三个代理将同时工作，提供全面的旅行推荐。


In [6]:
# 定义辅助函数，把不同形态的工作流输出统一整理成消息列表。
def collect_messages(outputs: list[Any]) -> list[Message]:
    # 初始化一个空列表，用来收集所有消息对象。
    messages: list[Message] = []
    # 遍历每一个工作流输出对象。
    for output in outputs:
        # 如果输出对象本身带有 messages 属性，就直接取出来。
        if hasattr(output, "messages"):
            messages.extend(output.messages)
        # 如果输出本身就是列表，也把其中的 Message 对象提取出来。
        elif isinstance(output, list):
            messages.extend([msg for msg in output if isinstance(msg, Message)])
    # 返回整理后的消息列表。
    return messages


# 定义异步函数，运行并发工作流并把三个代理的结果可视化展示出来。
async def display_travel_recommendations(destination: str):
    # 在 notebook 中先显示一块“正在处理”的状态提示。
    display(HTML(f"""
<div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
    <h3 style='margin: 0 0 10px 0; color: #e65100;'>🌍 正在为 {destination} 生成个性化旅行推荐</h3>
    <p style='margin: 0;'><strong>执行状态：</strong>3 个专业智能体正在并发处理任务，请稍候...</p>
</div>
    """))

    # 运行并发工作流，把目标城市作为用户请求传进去。
    events = await workflow.run(f"请为我提供 {destination} 的全面旅行推荐。")
    # 读取本次工作流的输出对象列表。
    outputs = events.get_outputs()

    # 如果确实拿到了输出，就继续解析和展示。
    if outputs:
        # 把输出统一转成消息列表。
        messages = collect_messages(outputs)
        # 只保留三个专业代理返回的消息。
        agent_responses = [
            msg for msg in messages
            # 根据当前条件决定后续走哪条逻辑分支。
            if msg.author_name in ["attractions_agent", "dining_agent", "history_agent"]
        ]

        # 展示总标题。
        display(HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h2 style='margin: 0 0 20px 0;'>{destination} 完整旅行指南</h2>
            <p style='margin: 0; font-size: 14px; opacity: 0.9;'>由 3 个专业智能体并发生成</p>
        </div>
        """))

        # 逐个处理不同代理的结构化输出。
        for response in agent_responses:
            # 尝试执行可能失败的解析或调用逻辑。
            try:
                # 处理景点代理的返回结果。
                if response.author_name == "attractions_agent":
                    # 把模型返回的 JSON 文本解析成结构化对象。
                    data = AttractionsRecommendation.model_validate_json(response.text)
                    display(HTML(f"""
                    <div style='padding: 20px; background: #e8f5e9; border-radius: 8px; margin: 15px 0; border-left: 4px solid #4caf50;'>
                        <h3 style='margin: 0 0 15px 0; color: #2e7d32;'>🏛️ 景点与旅行活动推荐</h3>
                        <p><strong>热门景点：</strong>{'、'.join(data.top_attractions) if data.top_attractions else '暂无'}</p>
                        <p><strong>推荐活动：</strong>{'、'.join(data.activities) if data.activities else '暂无'}</p>
                        <p><strong>最佳游览时间：</strong>{data.best_time_to_visit}</p>
                        <p><strong>交通建议：</strong>{data.transportation_tips}</p>
                    </div>
                    """))

                # 处理美食代理的返回结果。
                elif response.author_name == "dining_agent":
                    # 把模型返回的 JSON 文本解析成结构化对象。
                    data = DiningRecommendation.model_validate_json(response.text)
                    display(HTML(f"""
                    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
                        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🍽️ 美食与餐饮推荐</h3>
                        <p><strong>当地特色美食：</strong>{data.local_cuisine}</p>
                        <p><strong>必尝菜品：</strong>{'、'.join(data.must_try_dishes) if data.must_try_dishes else '暂无'}</p>
                        <p><strong>推荐餐厅：</strong>{'、'.join(data.recommended_restaurants) if data.recommended_restaurants else '暂无'}</p>
                        <p><strong>特色美食体验：</strong>{'、'.join(data.food_experiences) if data.food_experiences else '暂无'}</p>
                        <p><strong>用餐礼仪：</strong>{data.dining_etiquette}</p>
                    </div>
                    """))

                # 处理历史文化代理的返回结果。
                elif response.author_name == "history_agent":
                    # 把模型返回的 JSON 文本解析成结构化对象。
                    data = HistoryRecommendation.model_validate_json(response.text)
                    display(HTML(f"""
                    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
                        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>🏺 历史与文化介绍</h3>
                        <p><strong>历史意义：</strong>{data.historical_significance}</p>
                        <p><strong>文化亮点：</strong>{'、'.join(data.cultural_highlights) if data.cultural_highlights else '暂无'}</p>
                        <p><strong>重要历史时期：</strong>{'、'.join(data.important_periods) if data.important_periods else '暂无'}</p>
                        <p><strong>文化体验：</strong>{'、'.join(data.cultural_experiences) if data.cultural_experiences else '暂无'}</p>
                        <p><strong>趣味知识：</strong>{'、'.join(data.interesting_facts) if data.interesting_facts else '暂无'}</p>
                    </div>
                    """))
            # 如果某个代理的结果没法解析，就打印原始内容方便排查。
            except Exception as exc:
                print(f"Could not parse {response.author_name}: {exc}")
                print(response.text)


# 实际运行一次示例，查看 Barcelona 的并发推荐结果。
await display_travel_recommendations("巴塞罗纳")


# 第六步：测试案例 2 - 巴黎旅行推荐


In [7]:
# 等待异步操作完成，再继续执行后续代码。
await display_travel_recommendations("巴黎")


## 第七步：性能分析 - 并发与顺序

让我们测量并发执行与顺序执行之间的性能差异，以展示并发编排的优势。


In [8]:
# 定义异步函数 `measure_concurrent_performance`，用于处理需要 `await` 的流程。
async def measure_concurrent_performance(destination: str):
    """测量并发执行时间。"""
    start_time = time.time()
    # 启动工作流执行，并拿到本次运行的结果。
    events = await workflow.run(f"请为我提供 {destination} 的全面旅行推荐。")
    # 提取工作流最终产出的输出。
    outputs = events.get_outputs()
    end_time = time.time()
    message_count = len(collect_messages(outputs)) if outputs else 0
    # 返回当前函数的结果给调用方。
    return end_time - start_time, message_count


# 定义异步函数 `measure_sequential_performance`，用于处理需要 `await` 的流程。
async def measure_sequential_performance(destination: str):
    """测量顺序执行时间。"""
    # 创建顺序工作流构建器，让多个 Agent 按顺序执行。
    sequential_workflow = SequentialBuilder(
        participants=[attractions_agent, dining_agent, history_agent],
    # 根据前面配置生成最终可运行的工作流对象。
    ).build()

    start_time = time.time()
    # 启动工作流执行，并拿到本次运行的结果。
    events = await sequential_workflow.run(f"请为我提供 {destination} 的全面旅行推荐。")
    # 提取工作流最终产出的输出。
    outputs = events.get_outputs()
    end_time = time.time()
    message_count = len(collect_messages(outputs)) if outputs else 0
    # 返回当前函数的结果给调用方。
    return end_time - start_time, message_count


# 定义异步函数 `performance_comparison`，用于处理需要 `await` 的流程。
async def performance_comparison():
    """比较并发与顺序执行的性能."""
    test_destination = "巴塞罗纳"

    display(HTML("""
        <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
            <h3 style='margin: 0 0 10px 0; color: #e65100;'>⚡ 性能对比测试</h3>
            <p style='margin: 0;'>测试目的地：<strong>巴塞罗那（Barcelona）</strong></p>
        </div>
    """))

    concurrent_time, concurrent_msgs = await measure_concurrent_performance(test_destination)
    sequential_time, sequential_msgs = await measure_sequential_performance(test_destination)
    speedup = sequential_time / concurrent_time if concurrent_time > 0 else 0

    display(HTML(f"""
<div style='padding: 25px; background: linear-gradient(135deg, #2196f3 0%, #21cbf3 100%); color: white; border-radius: 12px; margin: 20px 0;'>
    <h2 style='margin: 0 0 20px 0;'>⚡ 性能测试结果</h2>

    <p><strong>并发执行：</strong>{concurrent_time:.2f} 秒，共 {concurrent_msgs} 条消息</p>

    <p><strong>串行执行：</strong>{sequential_time:.2f} 秒，共 {sequential_msgs} 条消息</p>

    <p><strong>性能提升：</strong>{speedup:.2f} 倍</p>
</div>
    """))


# 等待异步操作完成，再继续执行后续代码。
await performance_comparison()



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们对因使用此翻译而产生的任何误解或误读不承担责任。
